# 32 -- Section 01e Step 2: weights and hidden input

Computes x+ = v+ W+ + c+ and x- = v- W- + c- for the Rumble in the Bronx x h1
worked example, at the exact initial weights/biases used by notebook 31 (the
01c worked example), for two input scenarios: 01c (b = 0, so v+ = v- = a) and
01e (b = -1, so v+ = a + b and v- = a - b at the Rumble entry only). This
notebook does not touch the HTML; it only computes the numbers Step 2 needs.

## Part 0 -- Setup: scan notebooks/outputs, pick this notebook's number

In [1]:
from pathlib import Path
import json
import re
import math

import pandas as pd
import torch

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
notebooks_dir = root / "notebooks"
data_dir = root / "data"
out_dir = root / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

nb_paths = sorted(p for p in notebooks_dir.glob("*.ipynb") if "step2_hidden_input" not in p.stem)
num_re = re.compile(r"^(\d+)")
numbered = sorted(
    (int(m.group(1)), p) for p in nb_paths if (m := num_re.match(p.stem))
)
highest_num, highest_path = numbered[-1]
this_num = highest_num + 1
this_name = f"{this_num}_step2_hidden_input.ipynb"
print(f"highest existing notebook: {highest_num:>3d}  ({highest_path.name})")
print(f"this notebook is:          {this_num:>3d}  ({this_name})")

highest existing notebook:  31  (31_step8_minus_channel.ipynb)
this notebook is:           32  (32_step2_hidden_input.ipynb)


## Part 1 -- Reuse notebook 31's exact worked-example setup

Same train split, same movie (Rumble in the Bronx), same median user, same
seeds (plus channel seed 44, minus channel seed 45), same zero-initialized
hidden biases. Values are re-derived from `outputs/31_step8_results.json`
and the raw data, never typed by hand.

In [2]:
train_path = data_dir / "train_hyperbolic.csv"
test_path = data_dir / "test_hyperbolic.csv"
results31_path = out_dir / "31_step8_results.json"
for p in (train_path, test_path, results31_path):
    assert p.exists(), f"Missing {p}"

train_hyp = pd.read_csv(train_path)
test_hyp = pd.read_csv(test_path)
assert len(train_hyp) == 492055 and len(test_hyp) == 123474, "not the section 03 split"

results31 = json.loads(results31_path.read_text())
REAL_SEED = results31["seeds"]["plus_seed_reused"]
MINUS_SEED = results31["seeds"]["minus_seed_trained"]
N_HIDDEN = int(results31["architecture_match"]["128 hidden units"].split("=")[1])
H1_IDX = 0  # "h1" = first hidden unit, matching notebook 31
print(f"plus_seed={REAL_SEED}  minus_seed={MINUS_SEED}  N_HIDDEN={N_HIDDEN}")

plus_seed=44  minus_seed=45  N_HIDDEN=128


In [3]:
all_users = sorted(pd.concat([train_hyp["userId"], test_hyp["userId"]]).unique())
all_movies = sorted(pd.concat([train_hyp["movieId"], test_hyp["movieId"]]).unique())
n_movies = len(all_movies)
movie_to_col = {m: j for j, m in enumerate(all_movies)}

movies = pd.read_csv(data_dir / "movie.csv")
rumble_row = movies[movies["title"].str.contains("Rumble in the Bronx", case=False, na=False)]
assert len(rumble_row) == 1
RUMBLE_ID = int(rumble_row.iloc[0]["movieId"])
RUMBLE_TITLE = rumble_row.iloc[0]["title"]
rumble_col = movie_to_col[RUMBLE_ID]

rumble4 = train_hyp[(train_hyp["movieId"] == RUMBLE_ID) & (train_hyp["rating"] == 4.0)].copy()
train_counts = train_hyp.groupby("userId").size()
rumble4["total_count"] = rumble4["userId"].map(train_counts)
rumble4 = rumble4.sort_values(["total_count", "userId"]).reset_index(drop=True)
median_pos = (len(rumble4) - 1) // 2
median_row = rumble4.iloc[median_pos]
MEDIAN_USER = int(median_row["userId"])
assert MEDIAN_USER == results31["worked_example_plus"]["user"], "must match notebook 31's user"
print(f"movie: {RUMBLE_TITLE} (column={rumble_col})")
print(f"median user: {MEDIAN_USER}")

movie: Rumble in the Bronx (Hont faan kui) (1995) (column=27)
median user: 24127


In [4]:
user_rows = train_hyp[train_hyp["userId"] == MEDIAN_USER]
v0_full = torch.zeros(n_movies)
for row in user_rows.itertuples():
    v0_full[movie_to_col[row.movieId]] = row.a

a_val = v0_full[rumble_col].item()
assert abs(a_val - results31["worked_example_plus"]["v0"]) < 1e-9, "a must match notebook 31's v0"
print(f"a (Rumble, this user) = {a_val:.6f}")

g_init_p = torch.Generator().manual_seed(REAL_SEED)
W_plus_init = torch.randn(n_movies, N_HIDDEN, generator=g_init_p) * 0.01
c_plus_vec = torch.zeros(N_HIDDEN)

g_init_m = torch.Generator().manual_seed(MINUS_SEED)
W_minus_init = torch.randn(n_movies, N_HIDDEN, generator=g_init_m) * 0.01
c_minus_vec = torch.zeros(N_HIDDEN)

w_plus_rumble = W_plus_init[rumble_col, H1_IDX].item()
w_minus_rumble = W_minus_init[rumble_col, H1_IDX].item()
c_plus = c_plus_vec[H1_IDX].item()
c_minus = c_minus_vec[H1_IDX].item()
assert abs(w_plus_rumble - results31["worked_example_plus"]["w_before"]) < 1e-9
assert abs(w_minus_rumble - results31["worked_example_minus"]["w_before"]) < 1e-9
print(f"w+[Rumble,h1] = {w_plus_rumble:.6f}  w-[Rumble,h1] = {w_minus_rumble:.6f}")
print(f"c+[h1] = {c_plus}  c-[h1] = {c_minus}")

a (Rumble, this user) = 0.777778
w+[Rumble,h1] = 0.006433  w-[Rumble,h1] = -0.009362
c+[h1] = 0.0  c-[h1] = 0.0


## Part 2 -- x+ and x- for the 01c scenario (b = 0, v+ = v- = a)

Full matrix-vector product over all 1000 movies: `x = v @ W + c`, picking out
hidden unit h1. Sanity check: `sigmoid(x)` must match notebook 31's stored
`h0_1` for each channel, since that is exactly what notebook 31 computed.

In [5]:
x_plus_01c = (v0_full @ W_plus_init + c_plus_vec)[H1_IDX].item()
x_minus_01c = (v0_full @ W_minus_init + c_minus_vec)[H1_IDX].item()

sig = lambda z: 1 / (1 + math.exp(-z))
h_plus_check = sig(x_plus_01c)
h_minus_check = sig(x_minus_01c)
assert abs(h_plus_check - results31["worked_example_plus"]["h0_1"]) < 1e-6
assert abs(h_minus_check - results31["worked_example_minus"]["h0_1"]) < 1e-6
print(f"x+_01c = {x_plus_01c:.6f}   sigmoid(x+_01c) = {h_plus_check:.6f} (matches nb31 h0_1)")
print(f"x-_01c = {x_minus_01c:.6f}   sigmoid(x-_01c) = {h_minus_check:.6f} (matches nb31 h0_1)")

x+_01c = -0.211115   sigmoid(x+_01c) = 0.447416 (matches nb31 h0_1)
x-_01c = -0.205367   sigmoid(x-_01c) = 0.448838 (matches nb31 h0_1)


## Part 3 -- x+ and x- for the 01e scenario (b = -1, only the Rumble entry changes)

`b` is only defined for The Matrix and Rumble in the Bronx in this page's
worked examples, so only the Rumble entry of `v` is replaced: `a + b` feeds
the plus channel, `a - b` feeds the minus channel. Every other rated movie's
entry is unchanged, so the whole-vector dot product isolates exactly the
contribution of that one entry's change.

In [6]:
b_val = -1.0
channel_plus_01c = a_val
channel_minus_01c = a_val
channel_plus_01e = a_val + b_val
channel_minus_01e = a_val - b_val

v_plus_01e = v0_full.clone()
v_plus_01e[rumble_col] = channel_plus_01e
v_minus_01e = v0_full.clone()
v_minus_01e[rumble_col] = channel_minus_01e

x_plus_01e = (v_plus_01e @ W_plus_init + c_plus_vec)[H1_IDX].item()
x_minus_01e = (v_minus_01e @ W_minus_init + c_minus_vec)[H1_IDX].item()

print(f"channel+ input: 01c={channel_plus_01c:.4f}  01e={channel_plus_01e:.4f}")
print(f"channel- input: 01c={channel_minus_01c:.4f}  01e={channel_minus_01e:.4f}")
print(f"x+_01e = {x_plus_01e:.6f}")
print(f"x-_01e = {x_minus_01e:.6f}")

channel+ input: 01c=0.7778  01e=-0.2222
channel- input: 01c=0.7778  01e=1.7778
x+_01e = -0.217548
x-_01e = -0.214729


## Part 4 -- Save for Step 2 of section 01e

In [7]:
results = {
    "notebook": this_name,
    "reused_from": "31_step8_minus_channel.ipynb / outputs/31_step8_results.json",
    "movie": RUMBLE_TITLE,
    "user": MEDIAN_USER,
    "hidden_unit": "h1 (index 0)",
    "a": a_val,
    "b": b_val,
    "w_plus_rumble": w_plus_rumble,
    "w_minus_rumble": w_minus_rumble,
    "c_plus": c_plus,
    "c_minus": c_minus,
    "channel_plus_input": {"01c": channel_plus_01c, "01e": channel_plus_01e},
    "channel_minus_input": {"01c": channel_minus_01c, "01e": channel_minus_01e},
    "x_plus": {"01c": x_plus_01c, "01e": x_plus_01e},
    "x_minus": {"01c": x_minus_01c, "01e": x_minus_01e},
}
results_path = out_dir / "32_step2_hidden_input_results.json"
with results_path.open("w") as f:
    json.dump(results, f, indent=2)
print(f"saved {results_path}")
print(json.dumps(results, indent=2))

saved /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/outputs/32_step2_hidden_input_results.json
{
  "notebook": "32_step2_hidden_input.ipynb",
  "reused_from": "31_step8_minus_channel.ipynb / outputs/31_step8_results.json",
  "movie": "Rumble in the Bronx (Hont faan kui) (1995)",
  "user": 24127,
  "hidden_unit": "h1 (index 0)",
  "a": 0.7777777910232544,
  "b": -1.0,
  "w_plus_rumble": 0.006433217320591211,
  "w_minus_rumble": -0.009361893869936466,
  "c_plus": 0.0,
  "c_minus": 0.0,
  "channel_plus_input": {
    "01c": 0.7777777910232544,
    "01e": -0.2222222089767456
  },
  "channel_minus_input": {
    "01c": 0.7777777910232544,
    "01e": 1.7777777910232544
  },
  "x_plus": {
    "01c": -0.21111512184143066,
    "01e": -0.21754834055900574
  },
  "x_minus": {
    "01c": -0.20536701381206512,
    "01e": -0.2147289365530014
  }
}
